# YgoMaster-PackBuilder — Manifest Loader Tests

Notebook di test manuale/esplorativo per il **manifest layer**.

Obiettivi:
- verificare l'import del package `manifest`;
- individuare automaticamente la root della repository;
- caricare `config/project.yaml`;
- controllare la costruzione di `ProjectManifest`;
- ispezionare player, main pack, mini-pack e shared pack;
- verificare alcuni errori strutturali del loader.


## 1. Ambiente Python e root della repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository. "
        "Esegui il notebook dalla root del progetto oppure dalla cartella tests/."
    )

manifest_path = repo_root / "config" / "project.yaml"

print("Repository root:", repo_root)
print("Manifest path:", manifest_path)

assert manifest_path.exists(), f"Manifest non trovato: {manifest_path}"
print("Repository and manifest paths OK")

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Manifest path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\project.yaml
Repository and manifest paths OK


## 2. Import del manifest layer

In [2]:
from ygomaster_packbuilder.manifest import (
    ProjectManifest,
    load_project_manifest,
)

from ygomaster_packbuilder.manifest.models import (
    ManifestPack,
    ManifestPackSource,
    ManifestPlayer,
    ManifestProjectSettings,
    ManifestSharedPack,
)

print("Manifest imports OK")

Manifest imports OK


## 3. Caricamento di `config/project.yaml`

In [3]:
manifest = load_project_manifest(manifest_path)

assert isinstance(manifest, ProjectManifest)

print("Manifest loaded successfully")
manifest

Manifest loaded successfully


ProjectManifest(settings=ManifestProjectSettings(name='Master Duel Custom League', default_opening_policy='configuration-a'), players=[ManifestPlayer(key='livio', display_name='Marzius', main_pack=ManifestPack(key='livio-scaled_invasion', source=ManifestPackSource(provider='ygoprodeck', external_id='24116'), opening_policy=None), mini_packs=[ManifestPack(key='livio-scaled_invasion-molting_evolution', source=ManifestPackSource(provider='ygoprodeck', external_id='24176'), opening_policy=None), ManifestPack(key='livio-scaled_invasion-venomous_strike', source=ManifestPackSource(provider='ygoprodeck', external_id='24172'), opening_policy=None), ManifestPack(key='livio-scaled_invasion-alien_attack', source=ManifestPackSource(provider='ygoprodeck', external_id='24175'), opening_policy=None)]), ManifestPlayer(key='matteo', display_name='Error771', main_pack=ManifestPack(key='matteo-might_from_shine_and_stone', source=ManifestPackSource(provider='ygoprodeck', external_id='24083'), opening_polic

## 4. Project settings

In [5]:
assert isinstance(manifest.settings, ManifestProjectSettings)

print("Project name:", manifest.settings.name)
print("Default opening policy:", manifest.settings.default_opening_policy)

assert isinstance(manifest.settings.name, str)
assert manifest.settings.name.strip()

assert isinstance(manifest.settings.default_opening_policy, str)
assert manifest.settings.default_opening_policy.strip()

manifest.settings

Project name: Master Duel Custom League
Default opening policy: configuration-a


ManifestProjectSettings(name='Master Duel Custom League', default_opening_policy='configuration-a')

## 5. Player

Questa sezione controlla solo la struttura caricata dal manifest.

Le regole semantiche più forti — ad esempio *esattamente tre mini-pack per player* —
saranno responsabilità del validation layer.


In [6]:
assert isinstance(manifest.players, list)
assert len(manifest.players) > 0, "Il manifest non contiene player."

for player in manifest.players:
    assert isinstance(player, ManifestPlayer)
    assert isinstance(player.key, str)
    assert player.key.strip()
    assert isinstance(player.display_name, str)
    assert player.display_name.strip()
    assert isinstance(player.main_pack, ManifestPack)
    assert isinstance(player.mini_packs, list)

    print(f"Player: {player.key} ({player.display_name})")
    print(f"  Main pack: {player.main_pack.key}")
    print(f"  Mini-packs: {[pack.key for pack in player.mini_packs]}")
    print()

print("Player structures OK")

Player: livio (Marzius)
  Main pack: livio-scaled_invasion
  Mini-packs: ['livio-scaled_invasion-molting_evolution', 'livio-scaled_invasion-venomous_strike', 'livio-scaled_invasion-alien_attack']

Player: matteo (Error771)
  Main pack: matteo-might_from_shine_and_stone
  Mini-packs: ['matteo-might_from_shine_and_stone-full_house', 'matteo-might_from_shine_and_stone-promotion', 'matteo-might_from_shine_and_stone-furious_warmongers']

Player: lorenzo (Lem)
  Main pack: lorenzo-hotel_transilvania
  Mini-packs: ['lorenzo-hotel_transilvania-mini_1', 'lorenzo-hotel_transilvania-mini_2', 'lorenzo-hotel_transilvania-mini_3']

Player structures OK


## 6. Source dei pack

In [7]:
def check_source(source: ManifestPackSource) -> None:
    assert isinstance(source, ManifestPackSource)
    assert isinstance(source.provider, str)
    assert source.provider.strip()
    assert isinstance(source.external_id, str)
    assert source.external_id.strip()


for player in manifest.players:
    check_source(player.main_pack.source)

    for pack in player.mini_packs:
        check_source(pack.source)

for pack in manifest.shared_packs:
    check_source(pack.source)

print("All pack sources OK")

All pack sources OK


## 7. Main pack e mini-pack

Il loader non deve ancora derivare dati del domain come:
`PackType`, `expected_card_count`, `booster_size`, `owner_key`,
`parent_pack_key` o le carte.


In [8]:
for player in manifest.players:
    main_pack = player.main_pack

    assert isinstance(main_pack.key, str)
    assert isinstance(main_pack.source, ManifestPackSource)
    assert main_pack.opening_policy is None or isinstance(main_pack.opening_policy, str)

    for mini_pack in player.mini_packs:
        assert isinstance(mini_pack, ManifestPack)
        assert isinstance(mini_pack.key, str)
        assert isinstance(mini_pack.source, ManifestPackSource)
        assert mini_pack.opening_policy is None or isinstance(
            mini_pack.opening_policy, str
        )

print("Personal pack DTOs OK")

Personal pack DTOs OK


## 8. Shared packs

In [9]:
assert isinstance(manifest.shared_packs, list)

for pack in manifest.shared_packs:
    assert isinstance(pack, ManifestSharedPack)
    assert isinstance(pack.key, str)
    assert pack.key.strip()
    assert isinstance(pack.type, str)
    assert pack.type.strip()
    assert isinstance(pack.source, ManifestPackSource)
    assert pack.opening_policy is None or isinstance(pack.opening_policy, str)

    print(
        f"{pack.key}: type={pack.type}, "
        f"source={pack.source.provider}:{pack.source.external_id}"
    )

print("Shared pack structures OK")

shared_staples-traps_n_tricks: type=shared_staples, source=ygoprodeck:24110
Shared pack structures OK


## 9. Ispezione riassuntiva del manifest

In [10]:
print(f"Project: {manifest.settings.name}")
print(f"Default policy: {manifest.settings.default_opening_policy}")
print(f"Players: {len(manifest.players)}")
print(f"Shared packs: {len(manifest.shared_packs)}")
print()

for player in manifest.players:
    print(f"[{player.key}] {player.display_name}")
    print(
        f"  MAIN  {player.main_pack.key} -> "
        f"{player.main_pack.source.provider}:"
        f"{player.main_pack.source.external_id}"
    )

    for mini in player.mini_packs:
        print(
            f"  MINI  {mini.key} -> "
            f"{mini.source.provider}:{mini.source.external_id}"
        )

for shared in manifest.shared_packs:
    print(
        f"[SHARED] {shared.key} ({shared.type}) -> "
        f"{shared.source.provider}:{shared.source.external_id}"
    )

Project: Master Duel Custom League
Default policy: configuration-a
Players: 3
Shared packs: 1

[livio] Marzius
  MAIN  livio-scaled_invasion -> ygoprodeck:24116
  MINI  livio-scaled_invasion-molting_evolution -> ygoprodeck:24176
  MINI  livio-scaled_invasion-venomous_strike -> ygoprodeck:24172
  MINI  livio-scaled_invasion-alien_attack -> ygoprodeck:24175
[matteo] Error771
  MAIN  matteo-might_from_shine_and_stone -> ygoprodeck:24083
  MINI  matteo-might_from_shine_and_stone-full_house -> ygoprodeck:24171
  MINI  matteo-might_from_shine_and_stone-promotion -> ygoprodeck:24174
  MINI  matteo-might_from_shine_and_stone-furious_warmongers -> ygoprodeck:24173
[lorenzo] Lem
  MAIN  lorenzo-hotel_transilvania -> ygoprodeck:24119
  MINI  lorenzo-hotel_transilvania-mini_1 -> ygoprodeck:XXXXX
  MINI  lorenzo-hotel_transilvania-mini_2 -> ygoprodeck:XXXXX
  MINI  lorenzo-hotel_transilvania-mini_3 -> ygoprodeck:XXXXX
[SHARED] shared_staples-traps_n_tricks (shared_staples) -> ygoprodeck:24110


## 10. Controllo delle key dichiarate

Non è ancora il validation layer vero e proprio.
Questa cella raccoglie le key soltanto per aiutare a ispezionare il manifest corrente.


In [11]:
player_keys = [player.key for player in manifest.players]

pack_keys = []

for player in manifest.players:
    pack_keys.append(player.main_pack.key)
    pack_keys.extend(pack.key for pack in player.mini_packs)

pack_keys.extend(pack.key for pack in manifest.shared_packs)

print("Player keys:", player_keys)
print("Pack keys:", pack_keys)

print()
print("Unique player keys:", len(set(player_keys)), "/", len(player_keys))
print("Unique pack keys:", len(set(pack_keys)), "/", len(pack_keys))

Player keys: ['livio', 'matteo', 'lorenzo']
Pack keys: ['livio-scaled_invasion', 'livio-scaled_invasion-molting_evolution', 'livio-scaled_invasion-venomous_strike', 'livio-scaled_invasion-alien_attack', 'matteo-might_from_shine_and_stone', 'matteo-might_from_shine_and_stone-full_house', 'matteo-might_from_shine_and_stone-promotion', 'matteo-might_from_shine_and_stone-furious_warmongers', 'lorenzo-hotel_transilvania', 'lorenzo-hotel_transilvania-mini_1', 'lorenzo-hotel_transilvania-mini_2', 'lorenzo-hotel_transilvania-mini_3', 'shared_staples-traps_n_tricks']

Unique player keys: 3 / 3
Unique pack keys: 13 / 13


## 11. Test errore: file inesistente

Questo comportamento appartiene direttamente al loader,
quindi ci aspettiamo un `FileNotFoundError`.


In [12]:
missing_path = repo_root / "config" / "__manifest_that_does_not_exist__.yaml"

try:
    load_project_manifest(missing_path)
except FileNotFoundError as exc:
    print("Expected FileNotFoundError:", exc)
else:
    raise AssertionError(
        "Il loader avrebbe dovuto sollevare FileNotFoundError."
    )

Expected FileNotFoundError: YAML file not found: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\__manifest_that_does_not_exist__.yaml


## 12. Test errore: root YAML non valida

Il root del manifest deve essere un mapping YAML.
Una lista YAML non può rappresentare un `ProjectManifest`.


In [13]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    invalid_manifest = Path(tmpdir) / "invalid_root.yaml"
    invalid_manifest.write_text(
        "- first\n- second\n",
        encoding="utf-8",
    )

    try:
        load_project_manifest(invalid_manifest)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare un root YAML non-mapping."
        )

Expected ValueError: YAML file must be a mapping: C:\Users\LMDP1\AppData\Local\Temp\tmpaporn008\invalid_root.yaml
